<a href="https://colab.research.google.com/github/Alino4kaAlino4ka/TgBotGPT_IS_PPO/blob/main/TgBotGPT_IS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q aiogram openai==1.63.0 nest_asyncio aiosqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.4/804.4 kB 32.7 MB/s eta 0:00:00


In [ ]:
# @title Импорт библиотек и настройка окружения
import os
import asyncio
import nest_asyncio
import sqlite3
import json
import re
from datetime import datetime
from typing import Optional, Dict, List, Tuple
from aiogram import Bot, Dispatcher, types
from aiogram.filters import Command
from aiogram.types import InlineKeyboardMarkup, InlineKeyboardButton
from openai import OpenAI
from google.colab import userdata

# Применяем patch для работы asyncio в Colab
nest_asyncio.apply()

print("Библиотеки успешно импортированы и окружение настроено")

Библиотеки успешно импортированы и окружение настроено


In [ ]:
# @title Загрузка токенов из секретов Colab
try:
    TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN_INF_SYS')
    VSEGPT_API_KEY = userdata.get('VSEGPT')
except Exception as e:
    print("Ошибка загрузки токенов. Убедитесь, что Вы добавили их в секреты Colab:")
    print("1. Нажмите на значок ключа слева")
    print("2. Добавьте два секрета: TELEGRAM_BOT_TOKEN и VSEGPT_API_KEY")
    print(f"Ошибка: {str(e)}")
    raise

In [ ]:
# @title Инициализация Telegram и VSEGPT

# Инициализация клиента VseGPT
client = OpenAI(
    api_key=VSEGPT_API_KEY,
    base_url="https://api.vsegpt.ru/v1"
)

# Инициализация Telegram бота
bot = Bot(token=TELEGRAM_BOT_TOKEN)
dp = Dispatcher()

In [ ]:
# @title База знаний по дисциплине «Информационные системы»
# Составлено из РПУД для направления 44.03.02 Психолого-педагогическое образование

knowledge_base = {
    "тематика": "Информационные системы в образовании и психолого-педагогической деятельности",
    "количество записей": 20,
    "примеры вопросов": [
        "Что такое информационная система?",
        "Что такое HTML и для чего он нужен?",
        "Что такое интеллект-карта и как её создать?",
        "Что такое инфографика?",
        "Какие образовательные ресурсы есть в интернете?",
        "Что такое браузер?",
        "Что такое база данных?",
        "Что такое гипертекст?",
        "Как создать веб-страницу?",
        "Что такое корреляционный анализ?"
    ],
    "контент": {
        # ===== ТЕМА 1. ПОНЯТИЯ ИТ И ИС =====
        "Что такое информационная система?": (
            "Информационная система (ИС) — это система, совокупность, предназначенная для хранения, поиска, обработки и передачи информации, предполагающая соответствующие организационные ресурсы (человеческие, технические, финансовые),для сбора, обработки и распространения информацию (ISO/IEC 2382:2015).\n\n"
            "Состав ИС включает:\n"
            "• техническое обеспечение (аппаратные средства)\n"
            "• программное обеспечение\n"
            "• информационное обеспечение (базы данных)\n"
            "• организационное обеспечение (регламенты, инструкции)\n"
            "• кадровое обеспечение (специалисты)"
        ),
        "Что такое информационные технологии?": (
            "Информационные технологии (ИТ) — это система методов, способов,приемов, процессов и программно-технических средств, объединённых в технологическую цепочку, обеспечивающую сбор, обработку, хранение и передачу информации.\n\n"
            "Роль ИТ в образовании:\n"
            "• автоматизация учебного процесса\n"
            "• создание электронных образовательных ресурсов\n"
            "• проблемные и проектные методы в обучении\n"
            "• интерактивные методы преподавания и геймификация\n"
            "• разработка помощников на базе ИИ\n"
        ),
        "Что такое глобальная информационная технология?": (
            "Глобальная информационная технология (ГИТ) — технология, охватывающая процессы сбора, обработки и передачи информации на уровне всей организации или даже на межорганизационном уровне.\n\n"
            "Примеры: корпоративные информационные системы,электронные библиотеки, базы знаний для различных сфер, системы электронного документооборота, облачные платформы для образования."
        ),

        # ===== ТЕМА 2. СОЗДАНИЕ WEB-ИНТЕРФЕЙСА =====
        "Что такое HTML и для чего он нужен?": (
            "HTML (HyperText Markup Language) — это язык разметки гипертекста, предназначенный для создания веб-страниц.\n\n"
            "Основные принципы работы с HTML:\n"
            "1. HTML-документ состоит из тегов, которые определяют структуру страницы\n"
            "2. Теги записываются в угловых скобках: <тег>\n"
            "3. Большинство тегов имеют парный закрывающий тег: </тег>\n"
            "4. Структура документа включает <head> (заголовок) и <body> (тело)\n"
            "5. Для форматирования текста используются теги <h1>-<h6>, <p>, <b>, <i>, <u>\n"
            "6. Для создания списков — <ul> (маркированный) и <ol> (нумерованный)\n"
            "7. Для вставки изображений — <img>\n"
            "8. Для создания ссылок — <a>\n"
            "9. Для создания таблиц — <table>, <tr>, <td>"
        ),
        "Как создать веб-страницу на HTML?": (
            "Алгоритм создания веб-страницы на HTML:\n\n"
            "1. Открыть среду разработки (PyCharm, VS Code и др.)\n"
            "2. Создать файл с расширением .html\n"
            "3. Написать базовую структуру:\n"
            "   <!DOCTYPE html>\n"
            "   <html>\n"
            "   <head>\n"
            "      <title>Название страницы</title>\n"
            "   </head>\n"
            "   <body>\n"
            "      <h1>Заголовок</h1>\n"
            "      <p>Текст абзаца</p>\n"
            "   </body>\n"
            "   </html>\n"
            "4. Открыть файл в веб-браузере\n"
            "5. При необходимости использовать CSS для оформления и JavaScript для интерактивности"
        ),
        "Что такое конструктор сайтов?": (
            "Конструктор сайтов — это онлайн-сервис, позволяющий создавать веб-сайты без навыков программирования.\n\n"
            "Примеры конструкторов для образовательных целей:\n"
            "• Tilda — https://tilda.cc/ru/ — визуальный конструктор с готовыми шаблонами\n"
            "• Wix — универсальный конструктор\n"
            "• Google Sites — простой конструктор для учебных проектов\n"
            "• Readymag — для дизайнерских проектов\n\n"
            "В просессе освоения курса «Информационные системы» мы будем использовать Tilda для создания сайтов на профессиональную тему (психолого-педагогическую)."
        ),
        "Что такое гипертекст?": (
            "Гипертекст — это текст, содержащий ссылки на другие документы или фрагменты текста, выделенные цветом или подчёркиванием.\n\n"
            "Гиперссылка — это элемент гипертекста (слово, фраза, изображение или кнопка), при нажатии на который происходит переход на другой документ, страницу или фрагмент текста.\n\n"
            "В HTML гиперссылки создаются с помощью тега <a>:\n"
            "<a href='адрес_страницы'>Текст ссылки</a>\n\n"
            "Виды ссылок:\n"
            "• внешние — на другие сайты\n"
            "• внутренние — на разделы той же страницы\n"
            "• относительные — на файлы в той же папке\n"
            "• абсолютные — с полным URL-адресом"
        ),








        # ===== ТЕМА 3. ИНТЕЛЛЕКТ-КАРТЫ И ИНФОГРАФИКА =====
        "Что такое интеллект-карта?": (
            "Интеллект-карта (майнд-карта, mind map) — это метод визуализации информации, при котором центральная идея располагается в центре, а от неё отходят ветви с ключевыми понятиями и ассоциациями.\n\n"
            "Законы построения интеллект-карт:\n"
            "1. Центральная идея — в центре (образ, слово)\n"
            "2. Ветви — расходятся от центра, каждая ветвь — одна мысль\n"
            "3. Ключевые слова — на каждой ветви\n"
            "4. Использование цветов и изображений для ассоциаций\n"
            "5. Естественная структура — как дерево с ветвями\n\n"
            "Где применяются в образовании:\n"
            "• конспектирование лекций\n"
            "• подготовка к экзаменам\n"
            "• планирование проектов\n"
            "• мозговой штурм\n"
            "• структурирование материала по психолого-педагогическим темам"
        ),
        "Где создать интеллект-карту?": (
            "Бесплатные онлайн-сервисы для создания интеллект-карт:\n\n"
            "1. MindMup — https://app.mindmup.com — простой, без регистрации\n"
            "2. XMind — мощный инструмент с множеством шаблонов\n"
            "3. FreeMind — бесплатная программа для ПК\n"
            "4. Coggle — совместная работа над картами\n"
            "5. MindMeister — интеграция с Google Drive\n\n"
            "В просессе освоения курса «Информационные системы» мы будем использовать MindMup для создания интеллект-карт на профессиональную тему (психолого-педагогическую)."
        ),
        "Что такое инфографика?": (
            "Инфографика — это способ представления информации в графическом виде, сочетающий текст, изображения, диаграммы и графики, таблицы, рисунки для наглядного донесения сложных данных.\n\n"
            "Применение инфографики в образовательном процессе:\n"
            "• визуализация статистических данных\n"
            "• представление результатов исследований\n"
            "• создание наглядных учебных пособий\n"
            "• демонстрация процессов и алгоритмов\n"
            "• оформление презентаций и проектов\n\n"
            "Инструменты для создания инфографики:\n"
            "• Canva — https://www.canva.com/ru_ru/sozdat/infografika/\n"
            "• Piktochart\n"
            "• Visme"
        ),

        # ===== ТЕМА 4. ИНФОРМАЦИОННЫЕ РЕСУРСЫ ИНТЕРНЕТ =====
        "Какие есть образовательные ресурсы в интернете?": (
            "Образовательные ресурсы интернета классифицируются по типам:\n\n"
            "1. УЧЕБНЫЕ МАТЕРИАЛЫ:\n"
            "   • учебники и учебные пособия\n"
            "   • электронные учебные курсы\n"
            "   • тексты лекций\n"
            "   • лабораторные практикумы\n"
            "   • задачники\n"
            "   • методические рекомендации\n"
            "   • тесты и контрольные вопросы\n\n"
            "2. СПРАВОЧНЫЕ МАТЕРИАЛЫ:\n"
            "   • энциклопедии\n"
            "   • словари\n"
            "   • справочники\n"
            "   • базы данных\n"
            "   • геоинформационные системы\n\n"
            "3. ПЛАТФОРМЫ ДЛЯ ОБУЧЕНИЯ:\n"
            "   • Moodle (электронные курсы)\n"
            "   • Юрайт (электронные учебники)\n"
            "   • IPR Smart (библиотечная система)\n"
            "   • Stepik (онлайн-курсы)"
            "   • Portal RosNOU"
        ),
        "Что такое браузер?": (
            "Браузер — это программа для просмотра веб-страниц и навигации в сети Интернет.\n\n"
            "Основные функции браузера:\n"
            "• отображение HTML-страниц\n"
            "• обработка CSS-стилей\n"
            "• выполнение JavaScript-кода\n"
            "• управление вкладками\n"
            "• сохранение закладок\n"
            "• история посещений\n\n"
            "Популярные браузеры: Google Chrome, Mozilla Firefox, Microsoft Edge, Safari, Opera."
        ),
        "Что такое база данных?": (
            "База данных (БД) — это структурированная совокупность данных, организованных по определённым правилам.\n\n"
            "В простейшем виде БД — это таблица, где:\n"
            "• строки содержат объекты с их характеристиками (записи)\n"
            "• столбцы содержат однородные характеристики (поля)\n"
            "• первая строка содержит названия полей\n\n"
            "Виды баз данных:\n"
            "• реляционные (табличные) — SQL\n"
            "• документоориентированные — MongoDB\n"
            "• графовые — Neo4j\n"
            "• облачные — облачные хранилища данных"
        ),
        "Какие сервисы интернета существуют?": (
            "Основные сервисы глобальной сети Интернет:\n\n"
            "1. WWW (World Wide Web) — система гипертекстовых документов\n"
            "2. Электронная почта (E-mail) — обмен сообщениями\n"
            "3. FTP — передача файлов\n"
            "4. Поисковые системы — поиск информации\n"
            "5. Социальные сети — общение и обмен контентом\n"
            "6. Мессенджеры — мгновенный обмен сообщениями\n"
            "7. Облачные хранилища — хранение данных\n"
            "8. Онлайн-образование — дистанционное обучение"
        ),
        "Что такое электронная почта?": (
            "Электронная почта (E-mail) — это:\n\n"
            "1. Система пересылки сообщений между пользователями, в которой компьютер берёт на себя все функции по хранению и пересылке сообщений\n"
            "2. Обмен почтовыми сообщениями с любым абонентом сети Internet\n"
            "3. Средство связи через телефонные линии с помощью компьютерной сети\n"
            "4. Сетевая служба, позволяющая обмениваться текстовыми сообщениями и вложениями\n\n"
            "Современные возможности: пересылка документов HTML, изображений, файлов любых типов."
        ),

        # ===== КОРРЕЛЯЦИОННЫЙ АНАЛИЗ =====
        "Что такое корреляционный анализ?": (
            "Корреляционный анализ — это метод математической статистики, позволяющий установить взаимосвязи между изучаемыми явлениями.\n\n"
            "Основные понятия:\n"
            "• Корреляция — статистическая взаимосвязь между переменными\n"
            "• Коэффициент корреляции — числовая мера тесноты связи (от -1 до +1)\n"
            "• Положительная корреляция — при увеличении одной переменной растёт другая\n"
            "• Отрицательная корреляция — при увеличении одной переменной другая уменьшается\n\n"
            "Применение в психолого-педагогических исследованиях:\n"
            "• связь между успеваемостью и посещаемостью\n"
            "• влияние методик преподавания на результаты обучения\n"
            "• взаимосвязь психологических характеристик и академических показателей"
        ),
        "Что такое коэффициент Пирсона?": (
            "Коэффициент корреляции Пирсона (r) — это наиболее распространённый показатель линейной связи между двумя количественными переменными.\n\n"
            "Формула:\n"
            "r = Σ(x - x̄)(y - ȳ) / √(Σ(x - x̄)² · Σ(y - ȳ)²)\n\n"
            "где:\n"
            "• x и y — значения переменных\n"
            "• x̄ и ȳ — средние арифметические\n\n"
            "Интерпретация коэффициента:\n"
            "• |r| < 0.3 — связь слабая\n"
            "• 0.3 ≤ |r| < 0.7 — связь средняя\n"
            "• |r| ≥ 0.7 — связь сильная"
        ),
        "Что такое критерий Стьюдента?": (
            "Критерий Стьюдента (t-критерий) — это статистический метод для проверки гипотезы о различии средних значений двух выборок.\n\n"
            "Применение в педагогических исследованиях:\n"
            "• сравнение успеваемости экспериментальной и контрольной групп\n"
            "• оценка эффективности внедрения инновационной технологии\n"
            "• проверка значимости различий в психологических тестах\n\n"
            "Если t-критерий превышает критическое значение, различия считаются статистически значимыми."
        ),
        "Что такое критерий Манна-Уитни?": (
            "Критерий Манна-Уитни (U-критерий) — это непараметрический статистический тест, используемый для сравнения двух независимых выборок по уровню выраженности какого-либо признака.\n\n"
            "Применение:\n"
            "• когда данные не распределены нормально\n"
            "• для сравнения порядковых переменных\n"
            "• для малых выборок\n\n"
            "Используется в психолого-педагогических исследованиях при работе с ранговыми данными."
        ),

        # ===== ДОПОЛНИТЕЛЬНЫЕ ТЕРМИНЫ ИЗ ГЛОССАРИЯ (РАЗДЕЛ 6.1.1) =====
        "Что такое антивирус?": (
            "Антивирус — это программа, предназначенная для обнаружения и уничтожения компьютерных вирусов.\n\n"
            "Основные функции антивируса:\n"
            "• сканирование файлов\n"
            "• обнаружение вредоносного кода\n"
            "• удаление/карантин заражённых файлов\n"
            "• защита в реальном времени\n"
            "• автоматическое обновление вирусных баз"
        ),
        "Что такое архиватор?": (
            "Архиватор — это программа, предназначенная для сжатия файлов, помещения их в архив и записи полученного архива на диск.\n\n"
            "Функции архиватора:\n"
            "• сжатие данных (уменьшение размера)\n"
            "• разархивация (восстановление исходных файлов)\n"
            "• создание самораспаковывающихся архивов\n"
            "• защита архивов паролем\n\n"
            "Популярные архиваторы: WinRAR, 7-Zip, WinZip."
        ),
        "Что такое драйвер?": (
            "Драйвер — это программа, обеспечивающая правильную работу видеосистем и других устройств компьютера.\n\n"
            "Назначение драйверов:\n"
            "• обеспечение взаимодействия операционной системы с оборудованием\n"
            "• управление устройствами (принтер, видеокарта, звуковая карта)\n"
            "• перевод команд ОС в понятный устройству язык\n"
            "• обработка ошибок и диагностика устройств"
        ),
        "Что такое провайдер?": (
            "Провайдер — это компания, обеспечивающая доступ в Интернет по протоколу TCP/IP, доставку и хранение электронной почты.\n\n"
            "Услуги провайдера:\n"
            "• подключение к интернету\n"
            "• предоставление IP-адреса\n"
            "• хранение почтовых ящиков\n"
            "• поддержка клиентов\n"
            "• предоставление коммуникационных программ и драйверов"
        ),
        "Что такое модем?": (
            "Модем (модулятор-демодулятор) — это устройство, преобразующее цифровые сигналы в аналоговую форму и обратно для передачи их по линиям связи аналогового типа.\n\n"
            "Типы модемов:\n"
            "• проводные (DSL, кабельные)\n"
            "• беспроводные (Wi-Fi, мобильные)\n"
            "• спутниковые\n\n"
            "Современные модемы поддерживают высокие скорости передачи данных."
        ),
        "Что такое сканер?": (
            "Сканер — это устройство ввода текстовой и графической информации в компьютер путём оптического считывания информации.\n\n"
            "Типы сканеров:\n"
            "• планшетные — для документов и изображений\n"
            "• ручные — портативные\n"
            "• штрих-кодов — для считывания кодов\n"
            "• 3D-сканеры — для трёхмерных объектов\n\n"
            "Применение в образовании: оцифровка учебных материалов, создание электронных библиотек."
        ),
    }
}



In [ ]:
# @title Безопасность и системный промпт

FORBIDDEN_KEYWORDS = [
    "взломать", "взлом", "хак", "вредонос", "вирус",
    "обойти", "воров", "кража", "украсть",
    "незакон", "преступ", "мошен", "обман",
    "ддос", "атака", "порно", "эротик",
    "пират", "торрент", "читер", "эксплоит"
]

def is_forbidden(text: str) -> bool:
    text_lower = text.lower()
    for word in FORBIDDEN_KEYWORDS:
        if word in text_lower:
            return True
    return False

SYSTEM_PROMPT = (
    "Ты — ассистент преподавателя Алины Сергеевны Грибановой по дисциплине «Информационные системы» для студентов-психологов. Отвечай кратко, структурированно и понятно. Используй примеры из образования и психологии. Если вопрос не по теме, вежливо направь к учебным материалам.  НЕ используй markdown-разметку, только обычный текст с эмодзи для визуального оформления."
)

print("Настройки безопасности загружены")

Настройки безопасности загружены


In [ ]:
# @title Telegram bot

@dp.message(Command("start"))
async def start_command(message: types.Message):
    await message.answer(
        "👋 Привет! Я бот-помощник по дисциплине «Информационные системы».\n\n"
        "Моя цель — помочь студентам-психологам освоить основные понятия и технологии:\n"
        "• информационные системы и технологии\n"
        "• создание веб-страниц на HTML\n"
        "• интеллект-карты и инфографика\n"
        "• образовательные ресурсы интернета\n\n"
        "Задайте мне вопрос или введите /help для списка примеров."
    )

@dp.message(Command("help"))
async def help_command(message: types.Message):
    help_text = (
        "📚 База знаний по дисциплине «Информационные системы»:\n"
        f"🔹 Тематика: {knowledge_base['тематика']}\n"
        f"🔹 Количество записей: {knowledge_base['количество записей']}\n"
        "🔹 Примеры вопросов:\n"
    )
    for question in knowledge_base["примеры вопросов"]:
        help_text += f"  • {question}\n"
    help_text += "\n💡 Попробуйте задать один из этих вопросов или любой другой по теме!\n"
    help_text += "❓ Если ответа нет в базе, я обращусь к языковой модели."
    await message.answer(help_text)

@dp.message()
async def handle_message(message: types.Message):
    user_question = message.text.strip()

    # Проверка на запрещённые запросы
    if is_forbidden(user_question):
        await message.answer(
            "Извините, я не могу помочь с этим вопросом. "
            "Я создан для помощи в изучении информационных систем в образовательных целях. "
            "Если у вас есть вопросы по учебному материалу, я с радостью отвечу!"
        )
        return

    # Проверка базы знаний (поиск с частичным совпадением)
    answer = None
    for key, value in knowledge_base["контент"].items():
        if key.lower() in user_question.lower() or user_question.lower() in key.lower():
            answer = value
            break

    if answer:
        await message.answer(answer)
        return

    # Запрос к VseGPT API, если ответ не найден
    try:
        response = client.chat.completions.create(
            model='openai/gpt-4o-mini',
            messages=[
                {"role": "system", "content": (
                    "Ты — ассистент преподавателя Алины Сергеевны Грибановой по дисциплине «Информационные системы» для студентов-психологов. Отвечай кратко, структурированно и понятно. Используй примеры из образования и психологии. Если вопрос не по теме, вежливо направь к учебным материалам.  НЕ используй markdown-разметку, только обычный текст с эмодзи для визуального оформления."
                )},
                {"role": "user", "content": user_question}
            ],
            temperature=0.3,
            extra_headers={"X-Title": "Telegram Knowledge Bot IS"}
        )
        # Убираем возможный markdown из ответа (заменяем ** на пустоту, ` на '' и т.д.)
        reply = response.choices[0].message.content
        reply = reply.replace('**', '').replace('`', '').replace('*', '·')
        await message.answer(reply)
    except Exception as e:
        await message.answer("⚠️ Произошла ошибка при обработке запроса. Попробуйте позже.")

# ============================================================
# ЗАПУСК БОТА
# ============================================================

async def run_bot():
    print("🟢 Бот-помощник по дисциплине «Информационные системы» успешно запущен!")
    print("🔗 Перейдите в Telegram и начните общение с ботом")
    await dp.start_polling(bot)

def start_bot():
    try:
        loop = asyncio.get_event_loop()
        bot_task = loop.create_task(run_bot())
        loop.run_forever()
    except KeyboardInterrupt:
        print("\n🛑 Остановка бота...")
        bot_task.cancel()
        try:
            loop.run_until_complete(bot_task)
        except asyncio.CancelledError:
            print("✅ Бот успешно остановлен")
    except Exception as e:
        print(f"🔴 Критическая ошибка: {e}")
    finally:
        loop.run_until_complete(bot.session.close())
        loop.close()

start_bot()

🟢 Бот-помощник по дисциплине «Информационные системы» успешно запущен!
🔗 Перейдите в Telegram и начните общение с ботом


## Выводы по проведенному исследованию

В рамках учебной ознакомительной практики было проведено исследование, направленное на разработку интеллектуального Telegram-бота для сопровождения дисциплины «Информационные системы» в Домодедовском филиале РосНОУ. Работа выполнена на основе рабочей программы учебной дисциплины (РПУД) для направления 44.03.02 Психолого-педагогическое образование и учитывает уровень теоретической и практической подготовки студентов педагогов-психологов.

В ходе исследования была создана структурированная база знаний, включающая ключевые записи, охватывающие все основные темы дисциплины: понятия информационных технологий и систем, создание веб-интерфейсов на HTML, применение интеллект-карт и инфографики в образовании, образовательные ресурсы Интернета, а также глоссарий терминов и вопросы корреляционного анализа. Каждый ответ в базе знаний составлен с учётом профессиональной направленности студентов педагогов-психологов и содержит практические примеры из педагогической и психологической практики.

Техническая реализация проекта выполнена на платформе Google Colab с использованием асинхронного фреймворка aiogram для создания Telegram-бота и API VseGPT для генерации ответов на дополнительные вопросы, отсутствующие в локальной базе знаний. Код бота разделён на логические ячейки, что обеспечивает удобство запуска и модификации. Реализован механизм безопасности, блокирующий нежелательные запросы, что особенно важно при использовании в образовательной среде.

В процессе исследования были решены следующие задачи: проведён анализ рабочей программы дисциплины и выделены ключевые темы и термины; сформирована база знаний, адаптированная для студентов-психологов; спроектирована и реализована архитектура Telegram-бота с локальным поиском и интеграцией с языковой моделью; настроен интерфейс бота с учётом образовательных потребностей; подготовлен бот, готовый к апробации в учебном процессе.

Особую ценность представляет разработанный системный промпт, который позиционирует бота как ассистента преподавателя и ориентирует его на образовательные задачи. Бот не только отвечает на вопросы, но и побуждает студентов обращаться к учебным материалам, что способствует формированию самостоятельности в обучении.

Результаты исследования показывают, что использование Telegram-бота с локальной базой знаний и возможностью обращения к генеративным моделям может стать эффективным инструментом поддержки образовательного процесса. Бот позволяет снизить нагрузку на преподавателя по ответам на повторяющиеся вопросы, обеспечивает студентам  доступ к учебной информации и способствует более глубокому усвоению материала за счёт интерактивного формата взаимодействия.

Таким образом, разработанный прототип подтверждает потенциал использования искусственного интеллекта в образовании, что соответствует направлению магистерской диссертации «Исследование и анализ возможностей использования искусственного интеллекта в образовании». В перспективе проект может быть расширен за счёт добавления RAG-системы, персонализированных рекомендаций, системы отслеживания прогресса студентов и интеграции с корпоративными образовательными платформами филиала.

Разработанное решение представляет собой практический вклад в цифровизацию образовательного процесса Домодедовского филиала РосНОУ и может быть рекомендовано к внедрению в учебный процесс по дисциплине «Информационные системы» и в другие.